In [2]:
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [19]:

# -------- DailyDialog  dataset--------
TRAIN_PATH = "data/dailydialog/train.csv"
VAL_PATH = "data/dailydialog/validation.csv"


def load_dailydialog_csv(file_path):
    df = pd.read_csv(file_path)
    pairs = []

    for _, row in df.iterrows():
        dialog = row.get("dialog")

        if isinstance(dialog, str):
            # Extract quoted utterances
            turns = re.findall(r"'(.*?)'", dialog)
            turns = [t.strip() for t in turns if t.strip()]

            # Create input-response pairs
            for i in range(len(turns) - 1):
                pairs.append({
                    "input": turns[i],
                     "response": turns[i + 1]
                })

    return pairs


train_pairs = load_dailydialog_csv(TRAIN_PATH)
val_pairs = load_dailydialog_csv(VAL_PATH)

print("Train pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))


Train pairs: 57648
Validation pairs: 5388


In [22]:
#Dataset class
from torch.utils.data import Dataset, DataLoader
class DialogueDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        input_text = pair["input"]
        target_text = pair["response"]

        inputs = self.tokenizer(
            input_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        targets = self.tokenizer(
            target_text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": targets["input_ids"].squeeze()
        }

In [27]:
MODEL_PATH = "models/t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 17067.49it/s]


In [34]:
train_pairs = train_pairs[:3000]
val_pairs = val_pairs[:500]

In [35]:
#prepare dataloaders
train_dataset = DialogueDataset(train_pairs, tokenizer)
val_dataset = DialogueDataset(val_pairs, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [36]:
print("Train pairs:", len(train_pairs))
print("Dataset size:", len(train_dataset))
print("Batches:", len(train_loader))

Train pairs: 3000
Dataset size: 3000
Batches: 375


In [37]:
#training setup
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [39]:
epochs = 1

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

        if i % 200 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} COMPLETE - Avg Loss: {avg_loss:.4f}")

Epoch 1 | Batch 0/375 | Loss: 0.7718


KeyboardInterrupt: 